## 1 卷积和池化层
### 1.1 理论计算题
输入一张大小为 $3 \times 32 \times 32$（通道数×高×宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$。设定填充（Padding）为 2，步幅（Stride）为 2。
1. 请计算该卷积层输出的特征图（Feature Map）的尺寸（通道数×高×宽）。
2. 计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行多少次点乘（乘法）操作？


1. 假设输入图像尺寸为$n_h \times n_w$，卷积核尺寸为$k_h \times k_w$，填充为$p_h \times p_w$，步幅为$s_h \times s_w$，卷积输出尺寸计算公式为：

$$
H_{out} = \left\lfloor \frac{H_{in} - k_{h} + 2p_{h}}{s_{h}} \right\rfloor + 1
$$
$$
W_{out} = \left\lfloor \frac{W_{in} - k_{w} + 2p_{w}}{s_{w}} \right\rfloor + 1
$$

代入$n_h=32$，$n_w=32$，$k_h=5$，$k_w=5$，$p_h=2$，$p_w=2$，$s_h=2$，$s_w=2$，得到输出特征图的尺寸为：

$$
H_{out} = \left\lfloor \frac{32 - 5 + 2\times 2}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 16
$$
$$
W_{out} = \left\lfloor \frac{32 - 5 + 2\times 2}{2} \right\rfloor + 1 = 16
$$

又卷积核数量为16，所以输出特征图的通道数为16。因此，输出特征图的尺寸为$16 \times 16 \times 16$（通道数×高×宽）。

2. 单个输出通道的一个像素值，由输入对应区域与卷积核逐元素相乘后求和得到。   
**乘法次数等于卷积核的总元素个数**，即$3 \times 5 \times 5 = 75$次。

### 1.2 编程题
不使用深度学习框架的底层 Pooling API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。


In [1]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    """
    二维最大池化前向传播（支持多通道和批处理）。
    
    参数：
        x: np.ndarray, 形状为 (N, C, H, W) 的输入张量
        kernel_size: int 或 tuple (kH, kW)，池化窗口大小
        stride: int 或 tuple (sH, sW)，步幅，默认 1
        padding: int 或 tuple (pH, pW)，填充大小，默认 0
    
    返回：
        out: np.ndarray, 池化后的输出，形状 (N, C, H_out, W_out)
    """
    # 统一参数格式
    if isinstance(kernel_size, int):
        kH = kW = kernel_size
    else:
        kH, kW = kernel_size
    if isinstance(stride, int):
        sH = sW = stride
    else:
        sH, sW = stride
    if isinstance(padding, int):
        pH = pW = padding
    else:
        pH, pW = padding

    N, C, H, W = x.shape

    # 计算输出尺寸
    H_out = (H + 2 * pH - kH) // sH + 1
    W_out = (W + 2 * pW - kW) // sW + 1

    # 对高和宽进行零填充（只填充最后两维）
    if pH > 0 or pW > 0:
        x_padded = np.pad(x, ((0, 0), (0, 0), (pH, pH), (pW, pW)), mode='constant')
    else:
        x_padded = x

    # 初始化输出
    out = np.zeros((N, C, H_out, W_out), dtype=x.dtype)

    # 滑动窗口取最大值
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * sH
            h_end = h_start + kH
            w_start = j * sW
            w_end = w_start + kW
            # 取出窗口，形状 (N, C, kH, kW)，在最后两个轴上取最大值
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))

    return out

In [ ]:
# 创建随机输入 (2, 3, 32, 32)
x = np.random.randn(2, 3, 32, 32)

# 池化：核 2x2，步幅 2，填充 0
out = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
print(out.shape)  

# 带填充的池化
out2 = max_pool2d_forward(x, kernel_size=3, stride=2, padding=1)
print(out2.shape) 

(2, 3, 16, 16)
(2, 3, 16, 16)


## 2 LeNet, AlexNet, VGG 和 NiN
### 2.1 理论计算题
在 VGG 网络中，作者频繁使用多个 $3 \times 3$ 卷积核级联来代替较大的卷积核（如 $5 \times 5$ 或 $7 \times 7$）。假设输入和输出的特征图通道数均为 $C$。
1. 计算一个 $5 \times 5$ 卷积层（不带偏置）的参数量。
2. 计算两个串联的 $3 \times 3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量。


不带偏置的卷积层参数量 = $C_{in} \times C_{out} \times k_h \times k_w$
1. $5 \times 5$ 卷积层的参数量为 $C \times C \times 5 \times 5 = 25C^2$
2. 两个 $3 \times 3$ 卷积层的总参数量   
第一层 $C \times C \times 3 \times 3 = 9C^2$
第二层 $C \times C \times 3 \times 3 = 9C^2$
总参数量 $= 18C^2$    
对比：$25C^2$（单层 $5 \times 5$） vs. $18C^2$（两层 $3 \times 3$），使用两个 $3 \times 3$ 卷积层的参数量更少，同时保持相同的感受野（Receptive Field）。

### 2.2 编程题
NiN 网络的核心创新是引入了“1x1 卷积”组成的 NiN 块来代替传统的全连接层，以减少参数量。请使用 PyTorch（`torch.nn.Sequential`）定义一个标准的 NiN 块（NiN Block）。

要求：NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通的卷积层（指定窗口大小 `kernel_size`，步幅 `stride`，填充 `padding`）以及两个随后的 $1 \times 1$ 卷积层级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。


In [3]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    实现标准NiN块
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的核大小
        stride: 第一个卷积层的步幅
        padding: 第一个卷积层的填充
    返回:
        nn.Sequential: NiN块
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
    )

In [4]:
block = nin_block(in_channels=3, out_channels=64, kernel_size=5, stride=1, padding=2)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print(out.shape) 

torch.Size([1, 64, 32, 32])


该块通过两层 1×1 卷积增强了非线性表达能力，同时保持空间尺寸不变。

## 3 Inception, 批量归一化和残差网络
### 3.1 理论计算题
在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_{1}=2$，$x_{2}=4$，$x_{3}=6$，$x_{4}=8$。假设当前批量归一化层学到的缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$。

请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 $y_{1}$，$y_{2}$，$y_{3}$，$y_{4}$。      



从形式上来说，用 $x \in \mathcal{B}$表示一个来自小批量的输入，批量规范化BN根据以下表达式转换：     
$$\mathrm{BN}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \hat{\mu}_{\mathrm{B}}}{\hat{\sigma}_{\mathrm{B}}} + \beta.$$
其中，$\hat{\mu}_B$ 和 $\hat{\sigma}^2_B$ 分别是小批量 $\mathcal{B}$ 的均值和方差，计算公式如下：
$$\hat{\mu}_B = \frac{1}{|B|} \sum_{x \in B} x, \quad \hat{\sigma}^2_B = \frac{1}{|B|} \sum_{x \in B} (x - \hat{\mu}_B)^2 + \epsilon.$$
1. 计算小批量的均值：
$$\hat{\mu}_B = \frac{2 + 4 + 6 + 8}{4} = 5.$$
2. 计算小批量的方差（$\epsilon=0$,使用有偏估计）：
$$\hat{\sigma}^2_B = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} + 0 = \frac{9 + 1 + 1 + 9}{4} = 5.$$
3. 归一化：
$$\hat{x}_i = \frac{x_i - \hat{\mu}_B}{\sqrt{\hat{\sigma}^2_B}} = \frac{x_i - 5}{\sqrt{5}}.$$
4. 缩放和平移($\gamma=2$,$\beta=1$)：
$$y_{i} = 2 \cdot \frac{x_{i} - 5}{\sqrt{5}} + 1.$$
分别代入 $x_i$，最终输出值为：
$$y_{1} = 2 \cdot \frac{2 - 5}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}} \approx -1.683,$$
$$y_{2} = 2 \cdot \frac{4 - 5}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}} \approx 0.106,$$
$$y_{3} = 2 \cdot \frac{6 - 5}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1.894,$$
$$y_{4} = 2 \cdot \frac{8 - 5}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}} \approx 3.683$$


### 3.2 编程题
残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

要求：该块包含两个具有相同输出通道数的 $3 \times 3$ 卷积层，每个卷积层后跟一个批量归一化层。如果 `use_1x1conv=True`，则需要对输入应用一个 $1 \times 1$ 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输出进行按元素相加（$f(x)+x$）。


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residual(nn.Module):
    """
    实现ResNet标准残差块
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        # 第一个3×3卷积层，输出out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
       
        # 第二个3×3卷积层，输入输出均为out_channels（保持尺寸不变）
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            # 1x1卷积层，用于改变通道数和步长，使其与残差分支输出可直接相加
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.downsample = None
        
    def forward(self, x):
        identity = x
        # 残差分支
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)          # 第一个卷积后的激活

        out = self.conv2(out)
        out = self.bn2(out)

        # 调整输入（如果需要）
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity            # 跨层连接
        out = F.relu(out)          # 相加后的激活
        return out


In [8]:
# 测试代码
if __name__ == "__main__":
    # 输入通道 3，输出通道 64，不使用 1x1 卷积（需要输入输出通道相同）
    # 注意：当 in_channels != out_channels 时必须 use_1x1conv=True
    x = torch.randn(2, 3, 32, 32)
    block = Residual(3, 64, use_1x1conv=True)
    out = block(x)
    print(out.shape)
    
    # 通道不变时可不使用 1x1 卷积
    block2 = Residual(64, 64, use_1x1conv=False)
    out2 = block2(out)
    print(out2.shape)

torch.Size([2, 64, 32, 32])
torch.Size([2, 64, 32, 32])



## 4 图像增广, 微调和样式迁移
### 4.1 理论计算题
在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如 ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。请回答以下关于微调理论的问题：
1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？


1. 差异化学习率与层冻结的底层逻辑      
在微调过程中对“底层特征提取层”和“顶层输出层”采取不同的学习策略，主要是基于神经网络层级特征的分布规律以及防止“灾难性遗忘”的需要：     

*   **底层特征提取层（小学习率或冻结）**：预训练模型的浅层通常已经学会了捕捉通用的、基础的特征（如图像的边缘、纹理、形状等）。这些通用知识对于绝大多数下游任务都是适用的。如果对这些层使用较大的学习率进行剧烈更新，模型极易破坏已学到的通用特征表示，导致“灾难性遗忘”。因此，采用较小的学习率进行温和调整或直接将其参数固定（冻结），可以有效保护预训练权重，避免干扰其已学内容。
*   **顶层输出层（大学习率）**：由于新任务的分类目标与源数据集不同，新添加的顶层输出层通常是随机初始化的，缺乏先验知识。设置较大的学习率可以使其快速收敛，迅速调整权重以匹配底层特征提取器的输出，并适应新的分类目标。

2. “数据少且高度相似”场景下的防过拟合策略     
当目标数据集非常小，且与源数据集（如 ImageNet）高度相似时，模型完全有能力利用现有的底层特征来完成新任务。此时应采取以下策略来最大程度防止过拟合：    

*   **仅微调最后 1-2 层**：最核心的策略是冻结模型前面的绝大部分层（将其作为固定的特征提取器），只训练最后的全连接层或分类头。因为模型前面所有层提取的通用特征在当前任务中完全适用，无需重新学习。这不仅能最大程度防止在小样本上过拟合，还能大幅加快训练速度。
*  若仍需微调部分底层，使用**极小的学习率**，并配合**强正则化**如： L2 正则化（权重衰减）、Dropout（随机屏蔽部分神经元）等技术，进一步限制模型的复杂度。
*   **数据增广**：通过随机旋转、平移、缩放等变换，增加训练数据的多样性，提高模型的泛化能力。
*   **采用早停机制（Early Stopping）**：密切监控验证集的性能表现，一旦发现验证集损失不再下降甚至开始上升，即提前终止训练，防止模型去死记硬背少量训练数据中的噪声。


### 4.2 编程题
图像增广能有效增强模型的泛化能力。请利用 `torchvision.transforms` 模块创建一个组合图像增广管道（Pipeline）。
1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后的图像缩放到 $224 \times 224$ 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。


In [6]:
from torchvision import transforms

def get_augmentation_pipeline():
    """
    创建题目要求的图像增广管道
    """
    return transforms.Compose([
        # 1. 随机裁剪并缩放到224×224，面积比例0.08-1.0
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        # 2. 50%概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        # 3. 随机改变亮度、对比度、饱和度，变化范围0.5
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        # 4. 转换为PyTorch张量
        transforms.ToTensor()
    ])

# 测试代码
if __name__ == "__main__":
    from PIL import Image
    import numpy as np
    
    # 创建测试图像
    img = Image.fromarray(np.random.randint(0, 255, (500, 500, 3), dtype=np.uint8))
    aug = get_augmentation_pipeline()
    out = aug(img)
    print("增广后张量形状:", out.shape)  # torch.Size([3, 224, 224])

增广后张量形状: torch.Size([3, 224, 224])



## 5 目标检测, 计算机视觉训练技巧
### 5.1 理论计算题
在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知图像中两个边界框（以 `[左上角 x, 左上角 y, 右下角 x, 右下角 y]` 格式表示）：
1. 真实框（Ground Truth）$A=[10,10,50,50]$
2. 预测框（Prediction Box）$B=[30,30,70,70]$

请计算边界框 A 和边界框 B 之间的 IoU 准确值。



IoU（Intersection over Union）计算公式为：
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$
首先计算两个边界框的交集（Intersection）和并集（Union）：
- 边界框 A 的左上角坐标为 (10, 10)，右下角坐标为 (50, 50)，宽度和高度均为 40。
- 边界框 B 的左上角坐标为 (30, 30)，右下角坐标为 (70, 70)，宽度和高度均为 40。      
交集部分的左上角坐标为 (30, 30)，右下角坐标为 (50, 50)，宽度和高度均为 20，因此交集面积为 $20 \times 20 = 400$。      
并集面积可以通过以下方式计算：
$$|A \cup B| = |A| + |B| - |A \cap B| = 40 \times 40 + 40 \times 40 - 400 = 2800.$$
因此，IoU 的准确值为：
$$IoU = \frac{400}{2800} = \frac{1}{7} \approx 0.1429.$$



### 5.2 编程题
在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（One-hot），若设置平滑因子 $\epsilon=0.1$，则对于 K 分类问题，真实标签对应的目标概率从 1 变为 $1-\epsilon$，其余错误类别的概率从 0 变为 $\frac{\epsilon}{K-1}$。

请实现一个计算标签平滑后交叉熵损失的函数。

In [11]:
import torch
import torch.nn.functional as F

def label_smoothing_loss(logits, targets, epsilon=0.1, reduction='mean'):
    """
    手动实现标签平滑后的交叉熵损失
    参数:
        logits: 模型输出，未经过softmax，形状为 (batch_size, num_classes)
        targets: 真实标签，形状为 (batch_size,)
        epsilon: 平滑因子，默认0.1
        reduction: 'mean' 返回平均损失，'sum' 返回总和，'none' 返回每个样本损失。
    返回:
        loss: 平均损失值
    """
    K = logits.size(1)
    # 计算 log softmax
    log_probs = F.log_softmax(logits, dim=1)  # (N, K)

    # 构造平滑后的目标分布
    # 初始化为 epsilon / (K - 1)
    smooth_target = torch.full_like(log_probs, epsilon / (K - 1))
    # 将真实标签位置设为 1 - epsilon
    smooth_target.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)

    # 计算负对数似然：- sum(q * log(p))  per sample
    loss_per_sample = - (smooth_target * log_probs).sum(dim=1)
    
    if reduction == 'mean':
        return loss_per_sample.mean()
    elif reduction == 'sum':
        return loss_per_sample.sum()
    else:
        return loss_per_sample 

In [12]:
if __name__ == "__main__":
    N, K = 4, 10
    logits = torch.randn(N, K)
    targets = torch.tensor([2, 5, 1, 9])
    loss = label_smoothing_loss(logits, targets, epsilon=0.1)
    print(f"Label smoothing loss: {loss.item():.4f}")

Label smoothing loss: 2.6274
